In [ ]:
import os
# 批量测试文档夹中的所有文档
def scan_documents(folder_path: str):
    """扫描文件夹并显示所有支持的文档类型"""
    supported_exts = {".pdf", ".doc", ".docx", ".wps", ".ppt", ".pptx", 
                      ".xls", ".xlsx", ".txt", ".rtf", ".html", ".htm"}
    
    files = []
    for filename in os.listdir(folder_path):
        ext = os.path.splitext(filename)[1].lower()
        if ext in supported_exts:
            full_path = os.path.join(folder_path, filename)
            size_kb = os.path.getsize(full_path) / 1024
            files.append((filename, ext, size_kb))
    
    print(f"发现 {len(files)} 个文档:")
    print("-" * 80)
    for idx, (name, ext, size) in enumerate(files, 1):
        print(f"{idx}. [{ext:5}] {name:50} ({size:.2f} KB)")
    
    return files

# 扫描文档库
files = scan_documents("健康提示资源库")

In [ ]:
# 测试解析 PDF 文档
test_file = r"健康提示资源库\1.中国公民健康素养—基本知识与技能（2024年版） 释义.pdf"

if os.path.exists(test_file):
    result = test_document_parsing(test_file)
else:
    print(f"文件不存在: {test_file}")

In [ ]:
import os
from tika import parser as tika_parser

def test_document_parsing(file_path: str):
    """
    测试 Tika 解析文档的功能。
    显示文档的元数据和提取的文本前 1000 个字符。
    """
    print(f"正在解析文件: {file_path}")
    print(f"文件大小: {os.path.getsize(file_path) / 1024:.2f} KB\n")
    
    # 使用 Tika 解析文档
    parsed = tika_parser.from_file(file_path)
    
    # 获取元数据
    metadata = parsed.get("metadata") or {}
    print("=" * 60)
    print("📋 文档元数据:")
    print("=" * 60)
    for key, value in list(metadata.items())[:10]:  # 只显示前10个元数据
        print(f"{key}: {value}")
    
    # 获取内容
    content = (parsed.get("content") or "").strip()
    print("\n" + "=" * 60)
    print("📄 提取的文本内容:")
    print("=" * 60)
    print(f"总字符数: {len(content)}")
    print(f"总行数: {len(content.splitlines())}")
    print("\n前 2000 字符预览:")
    print("-" * 60)
    print(content[:2000])
    print("-" * 60)
    
    # 检测分页符
    if "\f" in content:
        pages = content.split("\f")
        print(f"\n✅ 检测到分页符，共 {len(pages)} 页")
    else:
        print("\n⚠️  未检测到分页符，文档可能是连续文本")
    
    return parsed

In [ ]:
import re
SPECIFIC_PATTERNS = [
    r"\d+\s*(kcal|千卡|kg|千克|cm|分钟|分/|小时|天|周|月|次|组|%|％|mmHg)",
    r"≥|≤|>|<|≈|～|~",
    r"(每周|每月|每日|每天|每次|至少|不超过|不少于)",
    r"(1RM|VO2R|HRR|MET|kcal/周|次/组|组/周)"
]
SPECIFIC_RE = re.compile("|".join(SPECIFIC_PATTERNS))
def looks_specific(s: str) -> bool:
    return bool(SPECIFIC_RE.search(s))

In [ ]:
import json

def filter_jsonl(input_path, output_path, check_field="advice"):
    """
    读取 input.jsonl → 过滤 → 输出 clean.jsonl
    只有满足 looks_specific() 的行才会被保留
    """
    kept = 0
    removed = 0

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for idx, line in enumerate(fin, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                print(f"第 {idx} 行 JSON 解析失败，跳过")
                removed += 1
                continue

            value = obj.get(check_field, "")

            # ❗ 核心判定逻辑：满足正则 → 保留
            if isinstance(value, str) and looks_specific(value):
                fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
                kept += 1
            else:
                removed += 1

    print(f"清洗完成：保留 {kept} 条，删除 {removed} 条")


In [ ]:
filter_jsonl("atoms_1.中国公民健康素养—基本知识与技能（2024年版） 释义.pdf copy.jsonl", "clean.jsonl")


In [1]:
from query import *
import json

res = query(system_prompt="hello", content="hello")

In [3]:
# OCR 识别 PDF 文档
import fitz  # PyMuPDF
from PIL import Image
import pytesseract
import io

def ocr_pdf(pdf_path: str, output_txt: str = "ocr_output.txt"):
    """
    使用 OCR 识别 PDF 文档中的文字并保存到 txt 文件
    """
    print(f"正在处理: {pdf_path}")
    
    # 打开 PDF
    doc = fitz.open(pdf_path)
    all_text = []
    
    print(f"总页数: {len(doc)}")
    
    for page_num in range(len(doc)):
        print(f"正在处理第 {page_num + 1}/{len(doc)} 页...")
        page = doc[page_num]
        
        # 将页面转换为图片
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 提高分辨率
        img_data = pix.tobytes("png")
        img = Image.open(io.BytesIO(img_data))
        
        # 使用 pytesseract 进行 OCR（中文+英文）
        text = pytesseract.image_to_string(img, lang='chi_sim+eng')
        all_text.append(f"--- 第 {page_num + 1} 页 ---\n{text}\n")
    
    doc.close()
    
    # 合并所有文本
    res = "\n".join(all_text)
    
    # 保存到文件
    with open(output_txt, "w", encoding="utf-8") as f:
        f.write(res)
    
    print(f"\n✅ OCR 完成！")
    print(f"总字符数: {len(res)}")
    print(f"已保存到: {output_txt}")
    
    return res

# 测试 OCR
test_pdf = r"健康提示资源库\【试卷】26考研张宇四套卷（数一）.pdf"
res = ocr_pdf(test_pdf)

正在处理: 健康提示资源库\【试卷】26考研张宇四套卷（数一）.pdf
总页数: 8
正在处理第 1/8 页...
正在处理第 2/8 页...
正在处理第 3/8 页...
正在处理第 4/8 页...
正在处理第 5/8 页...
正在处理第 6/8 页...
正在处理第 7/8 页...
正在处理第 8/8 页...

✅ OCR 完成！
总字符数: 13793
已保存到: ocr_output.txt


In [2]:
# 将 OCR 文本（使用已存在变量 res）保存为 txt
text = res.strip() if isinstance(res, str) else ""
if not text:
    print("未找到可保存的 OCR 文本（变量 res 为空或不存在）。")
else:
    out_path = "ocr_output.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"已保存到 {out_path}，字符数：{len(text)}")

已保存到 ocr_output.txt，字符数：36
